# Final Grade Prediction - Data Preparation

This notebook prepares and normalizes the UCI and OULAD datasets for final grade prediction.

## Objectives:
1. Load and preprocess UCI Student Performance Dataset
2. Load and preprocess OULAD Dataset
3. Align common features between datasets
4. Normalize target variables (final grades)
5. Save prepared datasets ready for train/test split and model training


## 1. Import Libraries and Setup


In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")


Libraries imported successfully!


## 2. Load UCI Student Performance Dataset


In [2]:
# Find UCI dataset directory
# From final-grade-prediction/ we need to go up 4 levels to reach project root
base_paths_uci = [
    '../../../../datasets/uci-student-performance/',  # From final-grade-prediction/ (4 levels up)
    '../../../datasets/uci-student-performance/',    # From analysis/ (3 levels up)
    '../../datasets/uci-student-performance/',        # From charaka/ (2 levels up)
    'datasets/uci-student-performance/'               # From project root
]

uci_dir = None
for path in base_paths_uci:
    if os.path.exists(path):
        uci_dir = path
        break

if uci_dir is None:
    raise FileNotFoundError(f"Could not find UCI dataset directory. Tried: {base_paths_uci}")

print(f"UCI dataset directory: {os.path.abspath(uci_dir)}")


UCI dataset directory: /Users/charaka/Desktop/Projects/uom-student-performance-analytics/datasets/uci-student-performance


In [3]:
# Load UCI datasets (semicolon-separated)
df_uci_math = pd.read_csv(os.path.join(uci_dir, 'student-mat.csv'), sep=';')
df_uci_por = pd.read_csv(os.path.join(uci_dir, 'student-por.csv'), sep=';')

# Combine UCI datasets
# Note: 382 students appear in both, but we'll combine for more data
df_uci = pd.concat([df_uci_math, df_uci_por], ignore_index=True)

print(f"UCI Math dataset: {df_uci_math.shape[0]} rows × {df_uci_math.shape[1]} columns")
print(f"UCI Portuguese dataset: {df_uci_por.shape[0]} rows × {df_uci_por.shape[1]} columns")
print(f"UCI Combined dataset: {df_uci.shape[0]} rows × {df_uci.shape[1]} columns")
print(f"\nUCI columns: {list(df_uci.columns)}")


UCI Math dataset: 395 rows × 33 columns
UCI Portuguese dataset: 649 rows × 33 columns
UCI Combined dataset: 1044 rows × 33 columns

UCI columns: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


## 3. Prepare UCI Dataset for Final Grade Prediction


In [4]:
# Create a copy for processing
df_uci_processed = df_uci.copy()

# Target variable: G3 (final grade, 0-20 scale)
# We'll normalize it to 0-1 scale for consistency
df_uci_processed['final_grade'] = df_uci_processed['G3']
df_uci_processed['final_grade_normalized'] = df_uci_processed['G3'] / 20.0

# Select features that can be aligned with OULAD
# Based on cross-validation plan: gender, age, engagement, academic history, education, absences, support

# Demographics
df_uci_processed['gender'] = df_uci_processed['sex'].map({'M': 'Male', 'F': 'Female'})
df_uci_processed['age_numeric'] = df_uci_processed['age']

# Engagement (studytime is already numeric 1-4, we'll keep it)
df_uci_processed['engagement'] = df_uci_processed['studytime'] / 4.0  # Normalize to 0-1

# Academic history
df_uci_processed['academic_history'] = df_uci_processed['failures']

# Education background (average of mother's and father's education)
df_uci_processed['education_background'] = (df_uci_processed['Medu'] + df_uci_processed['Fedu']) / 2.0 / 4.0  # Normalize to 0-1

# Absences (normalize to 0-1, max absences is 93)
df_uci_processed['absences_normalized'] = df_uci_processed['absences'] / 93.0

# Support indicators (combine family and school support)
df_uci_processed['famsup_binary'] = (df_uci_processed['famsup'] == 'yes').astype(int)
df_uci_processed['schoolsup_binary'] = (df_uci_processed['schoolsup'] == 'yes').astype(int)
df_uci_processed['support_indicator'] = (df_uci_processed['famsup_binary'] + df_uci_processed['schoolsup_binary']) / 2.0

# Select final feature set for UCI
uci_features = [
    'gender',
    'age_numeric',
    'engagement',
    'academic_history',
    'education_background',
    'absences_normalized',
    'support_indicator',
    'final_grade',
    'final_grade_normalized'
]

df_uci_final = df_uci_processed[uci_features].copy()

print(f"UCI processed dataset: {df_uci_final.shape[0]} rows × {df_uci_final.shape[1]} columns")
print(f"\nUCI features: {list(df_uci_final.columns)}")
print(f"\nUCI target (final_grade) statistics:")
print(df_uci_final['final_grade'].describe())
print(f"\nMissing values:")
print(df_uci_final.isnull().sum())


UCI processed dataset: 1044 rows × 9 columns

UCI features: ['gender', 'age_numeric', 'engagement', 'academic_history', 'education_background', 'absences_normalized', 'support_indicator', 'final_grade', 'final_grade_normalized']

UCI target (final_grade) statistics:
count    1044.000000
mean       11.341954
std         3.864796
min         0.000000
25%        10.000000
50%        11.000000
75%        14.000000
max        20.000000
Name: final_grade, dtype: float64

Missing values:
gender                    0
age_numeric               0
engagement                0
academic_history          0
education_background      0
absences_normalized       0
support_indicator         0
final_grade               0
final_grade_normalized    0
dtype: int64


In [5]:
# Find OULAD dataset directory
# From final-grade-prediction/ we need to go up 4 levels to reach project root
base_paths_oulad = [
    '../../../../datasets/open-university-learning-analytics-dataset/',  # From final-grade-prediction/ (4 levels up)
    '../../../datasets/open-university-learning-analytics-dataset/',    # From analysis/ (3 levels up)
    '../../datasets/open-university-learning-analytics-dataset/',        # From charaka/ (2 levels up)
    'datasets/open-university-learning-analytics-dataset/'               # From project root
]

oulad_dir = None
for path in base_paths_oulad:
    if os.path.exists(path):
        oulad_dir = path
        break

if oulad_dir is None:
    raise FileNotFoundError(f"Could not find OULAD dataset directory. Tried: {base_paths_oulad}")

print(f"OULAD dataset directory: {os.path.abspath(oulad_dir)}")


OULAD dataset directory: /Users/charaka/Desktop/Projects/uom-student-performance-analytics/datasets/open-university-learning-analytics-dataset


In [6]:
# Load OULAD CSV files
student_info = pd.read_csv(os.path.join(oulad_dir, 'studentInfo.csv'))
student_assess = pd.read_csv(os.path.join(oulad_dir, 'studentAssessment.csv'))
assessments = pd.read_csv(os.path.join(oulad_dir, 'assessments.csv'))

# Load studentVle - it's split into multiple parts in studentVle_split directory
student_vle_dir = os.path.join(oulad_dir, 'studentVle_split')
if os.path.exists(student_vle_dir):
    # Load all parts and concatenate
    student_vle_parts = []
    for i in range(1, 7):  # Parts 1-6
        part_file = os.path.join(student_vle_dir, f'studentVle_part{i}.csv')
        if os.path.exists(part_file):
            student_vle_parts.append(pd.read_csv(part_file))
            print(f"Loaded studentVle_part{i}.csv: {student_vle_parts[-1].shape}")
    student_vle = pd.concat(student_vle_parts, ignore_index=True)
    print(f"Combined studentVle: {student_vle.shape}")
else:
    # Fallback: try single file (if it exists)
    student_vle_file = os.path.join(oulad_dir, 'studentVle.csv')
    if os.path.exists(student_vle_file):
        student_vle = pd.read_csv(student_vle_file)
    else:
        raise FileNotFoundError(f"Could not find studentVle files. Checked: {student_vle_dir} and {student_vle_file}")

print(f"\nstudentInfo: {student_info.shape}")
print(f"studentAssessment: {student_assess.shape}")
print(f"assessments: {assessments.shape}")
print(f"studentVle: {student_vle.shape}")


Loaded studentVle_part1.csv: (2000000, 6)
Loaded studentVle_part2.csv: (2000000, 6)
Loaded studentVle_part3.csv: (2000000, 6)
Loaded studentVle_part4.csv: (2000000, 6)
Loaded studentVle_part5.csv: (2000000, 6)
Loaded studentVle_part6.csv: (655280, 6)
Combined studentVle: (10655280, 6)

studentInfo: (32593, 12)
studentAssessment: (173912, 5)
assessments: (206, 6)
studentVle: (10655280, 6)


## 5. Calculate Final Grade from OULAD Assessments


In [7]:
# Merge assessment metadata to get weights and module/presentation info
student_assess_merged = student_assess.merge(
    assessments[['id_assessment', 'code_module', 'code_presentation', 'weight']],
    on='id_assessment',
    how='left'
)

# Convert numeric columns
student_assess_merged['score'] = pd.to_numeric(student_assess_merged['score'], errors='coerce')
student_assess_merged['weight'] = pd.to_numeric(student_assess_merged['weight'], errors='coerce')

# Filter out rows where score is NaN or missing module/presentation info
student_assess_valid = student_assess_merged[
    (student_assess_merged['score'].notna()) & 
    (student_assess_merged['weight'].notna()) &
    (student_assess_merged['code_module'].notna()) &
    (student_assess_merged['code_presentation'].notna())
].copy()

# Calculate weighted final grade per student with error handling
# Formula: final_grade = Σ(score × weight) / Σ(weight)
def calculate_weighted_avg(group):
    scores = group['score'].values
    weights = group['weight'].values
    
    # Check if we have valid scores and weights
    if len(scores) == 0 or not np.any(~np.isnan(scores)):
        return np.nan
    
    # Check if weights are available and sum to non-zero
    valid_mask = ~np.isnan(scores) & ~np.isnan(weights)
    if not np.any(valid_mask):
        return np.nan
    
    valid_scores = scores[valid_mask]
    valid_weights = weights[valid_mask]
    
    weight_sum = np.sum(valid_weights)
    if weight_sum > 0:
        # Use weighted average
        return np.average(valid_scores, weights=valid_weights)
    else:
        # Fall back to simple mean if weights sum to zero
        return np.nanmean(valid_scores)

final_grades = student_assess_valid.groupby(
    ['code_module', 'code_presentation', 'id_student']
).apply(calculate_weighted_avg).reset_index(name='final_grade')

# Filter out NaN final grades
final_grades = final_grades[final_grades['final_grade'].notna()].copy()

print(f"Calculated final grades for {len(final_grades)} student-module combinations")
print(f"\nFinal grade statistics:")
print(final_grades['final_grade'].describe())


Calculated final grades for 25820 student-module combinations

Final grade statistics:
count    25820.000000
mean        70.225401
std         17.061704
min          0.000000
25%         60.363636
50%         72.960000
75%         82.857143
max        100.000000
Name: final_grade, dtype: float64


## 6. Aggregate VLE Engagement Metrics


In [8]:
# Aggregate VLE clicks per student (engagement metric)
vle_engagement = student_vle.groupby(
    ['code_module', 'code_presentation', 'id_student']
)['sum_click'].sum().reset_index(name='total_vle_clicks')

# Normalize engagement (using 95th percentile as max to handle outliers)
max_clicks = vle_engagement['total_vle_clicks'].quantile(0.95)
vle_engagement['engagement'] = vle_engagement['total_vle_clicks'] / max_clicks
vle_engagement['engagement'] = vle_engagement['engagement'].clip(0, 1)  # Cap at 1

print(f"VLE engagement calculated for {len(vle_engagement)} student-module combinations")
print(f"\nEngagement statistics:")
print(vle_engagement['engagement'].describe())


VLE engagement calculated for 29228 student-module combinations

Engagement statistics:
count    29228.000000
mean         0.263309
std          0.280342
min          0.000211
25%          0.054968
50%          0.155893
75%          0.373130
max          1.000000
Name: engagement, dtype: float64


## 7. Prepare OULAD Dataset Features


In [9]:
# Start with studentInfo as base
df_oulad_processed = student_info.copy()

# Merge final grades
df_oulad_processed = df_oulad_processed.merge(
    final_grades,
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

# Merge VLE engagement
df_oulad_processed = df_oulad_processed.merge(
    vle_engagement[['code_module', 'code_presentation', 'id_student', 'engagement']],
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

# Fill missing engagement with 0 (students with no VLE activity)
df_oulad_processed['engagement'] = df_oulad_processed['engagement'].fillna(0)

# Normalize final grade to 0-1 scale (OULAD scores are 0-100)
df_oulad_processed['final_grade_normalized'] = df_oulad_processed['final_grade'] / 100.0

# Also normalize to 0-20 scale to match UCI
df_oulad_processed['final_grade_0_20'] = df_oulad_processed['final_grade'] / 100.0 * 20.0

# Prepare features aligned with UCI

# Demographics
# gender is already in the dataset

# Age: Convert age_band to numeric (approximate)
age_mapping = {
    '0-35': 30,
    '35-55': 45,
    '55<=': 60
}
df_oulad_processed['age_numeric'] = df_oulad_processed['age_band'].map(age_mapping)

# Engagement: already calculated from VLE

# Academic history
df_oulad_processed['academic_history'] = df_oulad_processed['num_of_prev_attempts']

# Education background: Convert highest_education to numeric (0-4 scale)
education_mapping = {
    'No Formal quals': 0,
    'Lower Than A Level': 1,
    'A Level or Equivalent': 2,
    'HE Qualification': 3,
    'Post Graduate Qualification': 4
}
df_oulad_processed['education_background'] = df_oulad_processed['highest_education'].map(education_mapping) / 4.0  # Normalize to 0-1

# Absences: Derive from low VLE activity (proxy for absences)
# Students with very low engagement might have high absences
df_oulad_processed['absences_normalized'] = 1 - df_oulad_processed['engagement']  # Inverse of engagement as proxy
df_oulad_processed['absences_normalized'] = df_oulad_processed['absences_normalized'].clip(0, 1)

# Support indicator: Use IMD band as proxy (lower deprivation = more support)
# Map IMD band to numeric (lower number = more support)
imd_mapping = {
    '0-10%': 0.1,
    '10-20%': 0.2,
    '20-30%': 0.3,
    '30-40%': 0.4,
    '40-50%': 0.5,
    '50-60%': 0.6,
    '60-70%': 0.7,
    '70-80%': 0.8,
    '80-90%': 0.9,
    '90-100%': 1.0
}
df_oulad_processed['support_indicator'] = 1 - df_oulad_processed['imd_band'].map(imd_mapping).fillna(0.5)  # Inverse as support

# Select final feature set for OULAD (aligned with UCI)
oulad_features = [
    'gender',
    'age_numeric',
    'engagement',
    'academic_history',
    'education_background',
    'absences_normalized',
    'support_indicator',
    'final_grade',
    'final_grade_normalized',
    'final_grade_0_20'
]

# Filter out rows with missing final_grade (students who didn't complete assessments)
df_oulad_final = df_oulad_processed[oulad_features].copy()
df_oulad_final = df_oulad_final[df_oulad_final['final_grade'].notna()].copy()

print(f"OULAD processed dataset: {df_oulad_final.shape[0]} rows × {df_oulad_final.shape[1]} columns")
print(f"\nOULAD features: {list(df_oulad_final.columns)}")
print(f"\nOULAD target (final_grade) statistics:")
print(df_oulad_final['final_grade'].describe())
print(f"\nMissing values:")
print(df_oulad_final.isnull().sum())


OULAD processed dataset: 25820 rows × 10 columns

OULAD features: ['gender', 'age_numeric', 'engagement', 'academic_history', 'education_background', 'absences_normalized', 'support_indicator', 'final_grade', 'final_grade_normalized', 'final_grade_0_20']

OULAD target (final_grade) statistics:
count    25820.000000
mean        70.225401
std         17.061704
min          0.000000
25%         60.363636
50%         72.960000
75%         82.857143
max        100.000000
Name: final_grade, dtype: float64

Missing values:
gender                    0
age_numeric               0
engagement                0
academic_history          0
education_background      0
absences_normalized       0
support_indicator         0
final_grade               0
final_grade_normalized    0
final_grade_0_20          0
dtype: int64


In [10]:
# Check feature alignment
uci_feature_set = set(df_uci_final.columns) - {'final_grade', 'final_grade_normalized'}
oulad_feature_set = set(df_oulad_final.columns) - {'final_grade', 'final_grade_normalized', 'final_grade_0_20'}

print("UCI features (excluding target):")
print(sorted(uci_feature_set))
print("\nOULAD features (excluding target):")
print(sorted(oulad_feature_set))
print("\nCommon features:")
print(sorted(uci_feature_set & oulad_feature_set))
print("\nFeatures only in UCI:")
print(sorted(uci_feature_set - oulad_feature_set))
print("\nFeatures only in OULAD:")
print(sorted(oulad_feature_set - uci_feature_set))


UCI features (excluding target):
['absences_normalized', 'academic_history', 'age_numeric', 'education_background', 'engagement', 'gender', 'support_indicator']

OULAD features (excluding target):
['absences_normalized', 'academic_history', 'age_numeric', 'education_background', 'engagement', 'gender', 'support_indicator']

Common features:
['absences_normalized', 'academic_history', 'age_numeric', 'education_background', 'engagement', 'gender', 'support_indicator']

Features only in UCI:
[]

Features only in OULAD:
[]


## 9. Final Data Quality Checks


In [11]:
print("=== UCI Dataset Summary ===")
print(f"Shape: {df_uci_final.shape}")
print(f"Missing values: {df_uci_final.isnull().sum().sum()}")
print(f"\nTarget variable (final_grade) range: [{df_uci_final['final_grade'].min():.2f}, {df_uci_final['final_grade'].max():.2f}]")
print(f"Target variable (final_grade_normalized) range: [{df_uci_final['final_grade_normalized'].min():.2f}, {df_uci_final['final_grade_normalized'].max():.2f}]")

print("\n=== OULAD Dataset Summary ===")
print(f"Shape: {df_oulad_final.shape}")
print(f"Missing values: {df_oulad_final.isnull().sum().sum()}")
print(f"\nTarget variable (final_grade) range: [{df_oulad_final['final_grade'].min():.2f}, {df_oulad_final['final_grade'].max():.2f}]")
print(f"Target variable (final_grade_normalized) range: [{df_oulad_final['final_grade_normalized'].min():.2f}, {df_oulad_final['final_grade_normalized'].max():.2f}]")
print(f"Target variable (final_grade_0_20) range: [{df_oulad_final['final_grade_0_20'].min():.2f}, {df_oulad_final['final_grade_0_20'].max():.2f}]")


=== UCI Dataset Summary ===
Shape: (1044, 9)
Missing values: 0

Target variable (final_grade) range: [0.00, 20.00]
Target variable (final_grade_normalized) range: [0.00, 1.00]

=== OULAD Dataset Summary ===
Shape: (25820, 10)
Missing values: 0

Target variable (final_grade) range: [0.00, 100.00]
Target variable (final_grade_normalized) range: [0.00, 1.00]
Target variable (final_grade_0_20) range: [0.00, 20.00]


## 10. Save Prepared Datasets


In [12]:
# Create output directory if it doesn't exist
output_dir = 'prepared_data'
os.makedirs(output_dir, exist_ok=True)

# Save UCI dataset
uci_output_path = os.path.join(output_dir, 'uci_final_grade_prepared.csv')
df_uci_final.to_csv(uci_output_path, index=False)
print(f"✓ UCI dataset saved to: {uci_output_path}")
print(f"  Shape: {df_uci_final.shape}")

# Save OULAD dataset
oulad_output_path = os.path.join(output_dir, 'oulad_final_grade_prepared.csv')
df_oulad_final.to_csv(oulad_output_path, index=False)
print(f"✓ OULAD dataset saved to: {oulad_output_path}")
print(f"  Shape: {df_oulad_final.shape}")

# Save feature information
feature_info = {
    'common_features': sorted(list(uci_feature_set & oulad_feature_set)),
    'uci_features': sorted(list(df_uci_final.columns)),
    'oulad_features': sorted(list(df_oulad_final.columns)),
    'target_variables': {
        'uci': ['final_grade', 'final_grade_normalized'],
        'oulad': ['final_grade', 'final_grade_normalized', 'final_grade_0_20']
    }
}

import json
feature_info_path = os.path.join(output_dir, 'feature_info.json')
with open(feature_info_path, 'w') as f:
    json.dump(feature_info, f, indent=2)
print(f"\n✓ Feature information saved to: {feature_info_path}")

print("\n=== Data Preparation Complete ===")
print("\nBoth datasets are now ready for:")
print("  1. Train/test split")
print("  2. Model training")
print("  3. Cross-dataset evaluation")


✓ UCI dataset saved to: prepared_data/uci_final_grade_prepared.csv
  Shape: (1044, 9)
✓ OULAD dataset saved to: prepared_data/oulad_final_grade_prepared.csv
  Shape: (25820, 10)

✓ Feature information saved to: prepared_data/feature_info.json

=== Data Preparation Complete ===

Both datasets are now ready for:
  1. Train/test split
  2. Model training
  3. Cross-dataset evaluation
